# 15 · Perfect-supplier replenishment: JEPA planner vs tuned textbook rules

The world has one shop, four products, and one perfect supplier: what the shop pays for arrives in full after the delivery lag. Only demand is uncertain, and the shop sees sales, not demand (ADR 0033, `src/perfect_supplier/README.md`).

**Arms.** Every arm runs closed-loop on its own copy of each eval world, with the same demand path. Comparisons are therefore paired world by world.

| arm | what it is |
| --- | --- |
| `jepa` | planner + JEPA quantile forecaster (the learned buyer) |
| `direct` | planner + the same network without the JEPA latent loss (ablation) |
| `planner_moving_average` | planner + 28-day moving-average normal forecast (no learning) |
| `planner_oracle` | planner + the true expected demand path and CV (ceiling; not a real policy) |
| `tuned_*` | the four textbook rules, each tuned **once** across 64 tune worlds |
| `default_order_up_to` | the engine's default (s,S) |
| `never_order` | floor |

**Score** is margin capture: Δequity / ((price − cost) × total demand). 1.0 means every demanded unit was sold with no fee and no holding cost.

Data lives under `ROOT`. Run the commands in the README first if it is empty.

In [ ]:
%matplotlib inline
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.perfect_supplier import data, analysis
from src.perfect_supplier.analysis import LABELS

ROOT = Path("runs/perfect-supplier")
pd.set_option("display.width", 200, "display.max_columns", 60)

## 1 · The summary frame

One row per (world, arm). It holds the world's parameters (`delivery_lag`, `holding_rate`, `cv`, `weekly_amp`, `capacity`, `initial_cash`), per-product `base_mean_p*` and `initial_on_hand_p*`, realised `mean_demand_p*`, `zero_demand_days_p*`, `mean_order_qty_p*`, `order_count_p*`, and `fill_rate_p*`. It also holds the outcome: `score`, `delta_equity`, `net_profit`, `revenue`, `fees`, `holding_cost`, `lost_units`, and `order_days`. `split` is `train` (behaviour log), `eval` (final comparison), or `shadow`.

In [ ]:
summary = data.load_summary(ROOT)
print(summary.groupby(["split", "method"]).size().rename("worlds").to_string())
summary.head(3).T

## 2 · Headline: paired comparison on the eval worlds

The baseline is the best tuned textbook rule by mean score. `diff_vs_baseline` is the mean world-by-world difference in score, with a 95% bootstrap interval. `win_rate` is the share of worlds where the arm beats the baseline.

In [ ]:
ev = summary[summary["split"] == "eval"]
tuned = [m for m in ev["method"].unique() if m.startswith("tuned_")]
BASELINE = ev[ev["method"].isin(tuned)].groupby("method")["score"].mean().idxmax()
print("baseline:", BASELINE)
table = analysis.paired_table(summary, BASELINE)
(table.assign(**{c: table[c] * 100 for c in ["diff_vs_baseline", "ci_low", "ci_high"]})
      .rename(columns={"diff_vs_baseline": "diff (pts)", "ci_low": "ci low (pts)", "ci_high": "ci high (pts)"})
      .round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
analysis.plot_paired(summary, BASELINE, arms=[a for a in table.index if a != "never_order"], ax=ax);

Pairwise against **every** tuned textbook rule, and JEPA against its own ablations. The planner + moving-average arm separates the planner's value from the forecast's value.

In [ ]:
m = analysis.score_matrix(summary).dropna()
rows = []
for a, b in [("jepa", t) for t in tuned] + [("jepa", "direct"), ("jepa", "planner_moving_average"), ("planner_oracle", "jepa")]:
    d = (m[a] - m[b]).to_numpy()
    lo, hi = analysis.bootstrap_ci(d)
    rows.append({"arm": a, "vs": b, "diff (pts)": d.mean() * 100, "ci low": lo * 100, "ci high": hi * 100, "win rate": (d > 0).mean()})
pd.DataFrame(rows).round(3)

## 3 · Where the margin goes

Each arm's lost margin is split into lost sales, order fees, and holding cost, each as a share of the margin on all demand.

In [ ]:
analysis.breakdown(summary).round(4)

## 4 · Which worlds the learned buyer wins

Mean paired difference to the baseline, cut by world parameters.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
arms = ["jepa", "planner_moving_average", "planner_oracle"] + [t for t in tuned if t != BASELINE]
for ax, feat in zip(axes.flat, ["cv", "delivery_lag", "holding_rate", "weekly_amp"]):
    analysis.plot_by_world_feature(summary, BASELINE, feat, arms, ax=ax)
fig.tight_layout()

## 5 · The forecaster

Training curves from `models/*_fit.json`. `pinball` is the censored loss on held-out **sales** (what the model trains on). `pinball_true_demand` is scored against true demand, which the model never sees. That number is inflated by stretches where a behaviour policy left the shelf empty for weeks: demand there cannot be identified from sales by any forecaster.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for arm in ["jepa", "direct"]:
    path = ROOT / "models" / f"{arm}_fit.json"
    if not path.exists():
        continue
    h = pd.DataFrame(json.loads(path.read_text())["history"])
    ax[0].plot(h["step"], h["pinball"], label=arm)
    ax[1].plot(h["step"], h["pinball_true_demand"], label=arm)
ax[0].set_title("val censored pinball (sales)"); ax[1].set_title("val pinball vs true demand")
for a in ax: a.legend(); a.set_xlabel("step")
h.iloc[-1:][["step", "pinball", "pinball_true_demand", "coverage"]]

## 6 · One world, every arm on its own run

Change `SEED`, `PRODUCT`, `ARMS`, and `DAYS`. By default the seed is the world where JEPA loses most to the baseline. Look there first for illogical decisions.

In [ ]:
diff = (m["jepa"] - m[BASELINE]).sort_values()
print("JEPA worst worlds:\n", (diff.head(5) * 100).round(2).to_string())
print("JEPA best worlds:\n", (diff.tail(5) * 100).round(2).to_string())
SEED = int(diff.index[0])
PRODUCT = 0
ARMS = ["jepa", BASELINE, "planner_oracle"]
DAYS = (0, 120)
ts = data.load_timeseries(ROOT, split="eval", method=ARMS, world_seed=SEED)
ev[ev["world_seed"] == SEED].set_index("method")[["score", "order_days", "fees", "holding_cost", "lost_units", "delivery_lag", "cv", "holding_rate"]].loc[ARMS]

In [ ]:
analysis.plot_world(ts, SEED, PRODUCT, ARMS, days=DAYS);

In [ ]:
analysis.plot_forecast(ts, SEED, PRODUCT, "jepa");

The JEPA arm's decision columns (`dec_*`) per day:
- `dec_review`, `dec_service`: the winning plan
- `dec_order_today`: whether that plan orders today
- `dec_score`: its imagined Δequity over 35 days
- `dec_score_best_wait`, `dec_score_best_order`: the best plan that waits vs the best that orders
- `dec_forecast_mean_7d`: the median forecast summed over the next 7 days

In [ ]:
cols = ["day", "product", "demand", "sales", "on_hand_close", "in_transit", "requested", "filled",
        "dec_review", "dec_service", "dec_order_today", "dec_score", "dec_score_best_wait", "dec_score_best_order", "dec_forecast_mean_7d"]
ts[(ts["method"] == "jepa") & (ts["product"] == PRODUCT)][cols].head(30)

## 7 · Shadow run: same state, every arm's decision

`split=shadow`: JEPA drives the shop. On every day the oracle planner, the moving-average planner, and the tuned textbook rules read the **same** observation, and what they would have requested is logged (`shadow_<arm>_requested`). Nothing they ask for is executed.

In [ ]:
sh = data.load_timeseries(ROOT, split="shadow")
if len(sh):
    shadow_cols = [c for c in sh.columns if c.startswith("shadow_")]
    agree = pd.DataFrame({
        c.removeprefix("shadow_").removesuffix("_requested"): {
            "orders on same day (share of driver order days)": ((sh[c] > 0) & (sh["requested"] > 0)).sum() / max((sh["requested"] > 0).sum(), 1),
            "mean units when ordering": sh.loc[sh[c] > 0, c].mean(),
            "order days per product-year": (sh[c] > 0).groupby([sh["world_seed"], sh["product"]]).sum().mean(),
        } for c in shadow_cols})
    agree["jepa (driver)"] = [1.0, sh.loc[sh["requested"] > 0, "requested"].mean(), (sh["requested"] > 0).groupby([sh["world_seed"], sh["product"]]).sum().mean()]
    display(agree.round(2))
    analysis.plot_shadow(sh, int(sh["world_seed"].iloc[0]), 0, days=(0, 90))

## 8 · The detailed time-series frame

Every simulation is stored, training and evaluation alike: one row per day × product, partitioned by `split` and `method` under `timeseries/`. Read any slice without loading the rest:

```python
data.load_timeseries(ROOT, split="eval", method="jepa", world_seed=[3000000, 3000001])
data.load_timeseries(ROOT, split="train", columns=["world_seed", "day", "product", "demand", "sales", "censored"])
```

Columns:
- **Per product:** `demand`, `demand_mean` (truth, never shown to policies), `sales`, `lost`, `censored`, `on_hand_open`, `arrivals`, `in_transit`, `position`, `requested`, `filled`, `on_hand_close`
- **Shop, repeated on each product row:** `cash_open`, `cash_close`, `purchase_cost`, `fee_paid`, `revenue`, `holding_cost`, `equity_close`
- **Decisions:** `dec_*` for each arm's decision columns, `shadow_*` for shadow requests
- **Training log only:** `behavior` names the policy that ran the shop

In [ ]:
print(sorted(p.name for p in (ROOT / "timeseries").glob("split=*/method=*")))
example = data.load_timeseries(ROOT, split="eval", method="jepa", world_seed=SEED)
print(example.shape)
example.head()